In [ ]:
# Reproducibility notes:
#   Python version: 3.13.3 (Clang 15.0.0)
#   pandas version: 2.2.3
#   numpy version:  2.2.6
#   requests version: 2.32.3
# To check your own environment (after running the imports in Block 1):
#   import sys; print(sys.version)
#   print(pd.__version__, np.__version__, requests.__version__)

# Block 1: Import Libraries and Define Shared Constants

# Import Libraries
import pandas as pd
import requests
import io
import numpy as np
import os


# Shared constants used throughout this notebook 

# NAICS industries under study
NAICS_CODES = ["11", "23", "42", "48_49", "56", "72"]

# All 32 Texas border counties as defined by Texas Department of State Health

Texas_Border_Counties = [
    48043, 48047, 48061, 48105, 48109, 48127, 48131, 48137, 48141, 48163,
    48215, 48229, 48243, 48247, 48261, 48271, 48283, 48311, 48323, 48371,
    48377, 48385, 48389, 48427, 48435, 48443, 48463, 48465, 48479, 48489,
    48505, 48507
]

# Control states: Arizona (4), New Mexico (35), California (6)
CONTROL_STATES = [4, 35, 6]

# Panel timeline
YEARS = range(2018, 2026)
QUARTERS = range(1, 5)
EXPECTED_PERIODS = [f"{y}Q{q}" for y in YEARS for q in QUARTERS]

# Treatment cutoff: Operation Lone Star border surge begins 2021 Q2
def is_post_treatment(year, qtr):
    return int((year > 2021) or (year == 2021 and qtr >= 2))

In [ ]:
# Block 2: Acquire QCEW Data 

# Define target NAICS industries
target_industries = [
    "11",    # Agriculture
    "23",    # Construction# Block 2: Acquire QCEW Data 
]
all_slices = []
print("Connecting to BLS Open Data Server...")
for naics in NAICS_CODES:
    for year in YEARS:
        for qtr in QUARTERS:
            url = f"https://data.bls.gov/cew/data/api/{year}/{qtr}/industry/{naics}.csv"
            try:
                response = requests.get(url, timeout=30)
                if response.status_code == 200:
                    df = pd.read_csv(
                        io.StringIO(response.text),
                        dtype={'area_fips': str, 'own_code': str, 'agglvl_code': str}
                    )
                    df = df[df['area_fips'].str.isnumeric()].copy()
                    df['area_fips_int'] = df['area_fips'].astype(int)
                    df['state_fips'] = df['area_fips_int'] // 1000
                    filtered_df = df[
                        (df['own_code'] == '5') &
                        (df['agglvl_code'] == '74') &
                        (df['area_fips_int'].isin(Texas_Border_Counties) | df['state_fips'].isin(CONTROL_STATES))
                    ].copy()
                    filtered_df['industry_run'] = naics
                    filtered_df['year'] = year
                    filtered_df['quarter'] = qtr
                    filtered_df['time_period'] = f"{year}Q{qtr}"
                    filtered_df['post_treatment'] = is_post_treatment(year, qtr)
                    all_slices.append(filtered_df)
                    print(f"Pulled & Filtered: NAICS {naics} -> {year} Q{qtr} ({len(filtered_df)} rows)")
                else:
                    print(f"No data: NAICS {naics} -> {year} Q{qtr} (HTTP {response.status_code})")
            except Exception as e:
                print(f"Error: NAICS {naics} -> {year} Q{qtr}: {e}")
if all_slices:
    clean_isolated_df = pd.concat(all_slices, ignore_index=True)
    print(f"\nPipeline verified. Master dataframe contains {len(clean_isolated_df)} clean rows.")
else:
    clean_isolated_df = pd.DataFrame()
    print("\nNo data was successfully downloaded.")

In [ ]:
# Block 3: Confirm the Treatment Group has all Quarters

# Check all 32 Texas border counties specifically
print("\n--- TREATMENT COUNTIES ---")
for fips in Texas_Border_Counties:
    county_data = clean_isolated_df[
        clean_isolated_df["area_fips_int"] == fips
    ]
    print(f"\nFIPS {fips}")
    if len(county_data) == 0:
        print("  COUNTY NOT FOUND IN DATASET")
        continue
    for naics in NAICS_CODES:
        observed = county_data[
            county_data["industry_run"] == naics
        ]["time_period"].unique()
        missing = sorted(set(EXPECTED_PERIODS) - set(observed))
        if len(missing) == 0:
            print(f"  NAICS {naics}: PERFECT — {len(observed)}/32 quarters")
        else:
            print(f"  NAICS {naics}: MISSING {len(missing)} quarters")
            print(f"     Missing: {missing}")

In [ ]:
# Block 4: Remove Counties from Treatment and Countrol Groups that are Missing Quarters and Save as New Master Dataframe

# Drop Counties with Data Suppression from the master dataframe
clean_isolated_df = clean_isolated_df[
    ~clean_isolated_df['area_fips_int'].isin([
        35028, 6003, 35021, 6091, 35019, 35033, 35011,  # control-county drops (AZ/NM/CA)
        48137, 48385, 48507, 48243, 48443, 48247, 48261, 48311, 48505  # treatment-county drops (Edwards, Real, Zavala, Jeff Davis, Terrell, Jim Hogg, Kenedy, McMullen, Zapata)
    ])
].copy()
print(f"Control and treatment counties with data suppression successfully dropped. New master dataframe size: {len(clean_isolated_df)} rows.")

In [ ]:
# Block 5: Save Master Dataframe Locally in case Project is Ininterupted 

# Save Industry Code as string and save dataset to hardrive
clean_isolated_df['industry_code'] = clean_isolated_df['industry_code'].astype(str)
clean_isolated_df.to_parquet("clean_isolated_df_6naics_32county.parquet")

print("Master dataframe saved locally")

In [ ]:
# Block 6: Re-Check All Counties in the Master Dataset to Confirm Cleaning was Successful

for naics in NAICS_CODES:
    ind_df = clean_isolated_df[clean_isolated_df["industry_run"] == naics]
    incomplete = []
    for fips, county_data in ind_df.groupby("area_fips_int"):
        observed = county_data["time_period"].unique()
        missing = sorted(set(EXPECTED_PERIODS) - set(observed))
        if len(missing) > 0:
            incomplete.append((fips, len(observed), missing))
    print(f"\nNAICS {naics}:")
    if len(incomplete) == 0:
        print("  PERFECT — every county has all 32 quarters.")
    else:
        print(f"  {len(incomplete)} counties have missing observations.")
        for fips, count, missing in incomplete:
            print(f"  • FIPS {fips} — {count}/32 quarters; missing: {missing}")

In [ ]:
# Block 7: Check for Differing County Data Coverage across NAICS Codes to Catch Data Gaps before Building the Analysis Dataset.

for naics in NAICS_CODES:
    counties = set(
        clean_isolated_df.loc[
            clean_isolated_df["industry_run"] == naics,
            "area_fips_int"
        ].unique()
    )
    print(f"NAICS {naics}: {len(counties)} counties")

print("\n--- DIFFERENCES ---")
county_sets = {}
for naics in NAICS_CODES:
    county_sets[naics] = set(
        clean_isolated_df.loc[
            clean_isolated_df["industry_run"] == naics,
            "area_fips_int"
        ].unique()
    )
for naics in NAICS_CODES:
    missing_from_this = (
        set.union(*county_sets.values()) - county_sets[naics]
    )
    print(f"\nNAICS {naics} is missing:")
    print(sorted(missing_from_this))

In [ ]:
#Block 8: Load the Cleaned Data into a Single Master File and Export SDiD Matrices

clean_isolated_df = pd.read_parquet("clean_isolated_df_6naics_32county.parquet")

# Map the targets: (raw QCEW column, new log-transformed column name)
metrics = {
    "sdid_exports_employment": ("month3_emplvl", "log_employment"),
    "sdid_exports_estabs": ("qtrly_estabs", "log_estabs"),
    "sdid_exports_wages": ("avg_wkly_wage", "log_wage")
}

for output_dir, (raw_col, log_col) in metrics.items():
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nProcessing panels for: {output_dir}...")

    summary_rows = []

    for naics in NAICS_CODES:
        ind_df = clean_isolated_df[clean_isolated_df['industry_run'] == naics].copy()
        ind_df[log_col] = np.log(ind_df[raw_col] + 1)

        Y_temp = ind_df.pivot(index='area_fips_int', columns='time_period', values=log_col)
        Y_temp = Y_temp.reindex(sorted(Y_temp.columns), axis=1)

        W_temp = pd.DataFrame(0, index=Y_temp.index, columns=Y_temp.columns)
        for hub in Texas_Border_Counties:
            if hub in W_temp.index:
                for col in W_temp.columns:
                    year, qtr = int(col[:4]), int(col[-1])
                    if is_post_treatment(year, qtr):
                        W_temp.loc[hub, col] = 1

        is_treated_unit = W_temp.sum(axis=1) > 0
        is_post_period = W_temp.sum(axis=0) > 0

        n_control = int((~is_treated_unit).sum())
        n_treated = int(is_treated_unit.sum())
        n_pre = int((~is_post_period).sum())
        n_post = int(is_post_period.sum())

        row_order = is_treated_unit.sort_values(kind="stable").index
        col_order = is_post_period.sort_values(kind="stable").index
        Y_sorted = Y_temp.loc[row_order, col_order]

        prefix = "Y_sorted_estabs_" if "estabs" in output_dir else ("Y_sorted_wages_" if "wages" in output_dir else "Y_sorted_")
        Y_sorted.to_csv(f"{output_dir}/{prefix}naics_{naics}.csv", index=True)

        summary_rows.append({
            "naics": naics,
            "N0_control_units": n_control,
            "N1_treated_units": n_treated,
            "T0_pre_periods": n_pre,
            "T1_post_periods": n_post,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(f"{output_dir}/sdid_config.csv", index=False)
    print(f"✓ Created sdid_config.csv and matrix slices for {output_dir}")

print("\nAll matrix exports and metadata configs completed successfully from the single Parquet file.")